# Cross-encoder re-scorer (Business Entity Resolution)
Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**.
Put the 6 data files in Google Drive folder **MyDrive/ber_ce/** first.
If Colab disconnects, just Run all again: it resumes from the saved checkpoint / score chunks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ber_ce'
import os; print(sorted(os.listdir(D)))
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
import os, time, math, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL = 'cross-encoder/ms-marco-MiniLM-L-12-v2'   # Apache-2.0, 33M parameters
MAXLEN = 96
TRAIN_MINUTES = 70          # training time budget
CK = f'{D}/ce_model'
try:
    tok = AutoTokenizer.from_pretrained(MODEL)
except Exception as e:
    print('fallback model', e); MODEL = 'cross-encoder/ms-marco-MiniLM-L12-v2'; tok = AutoTokenizer.from_pretrained(MODEL)
print('model', MODEL)
def load(name):
    rec = pd.read_parquet(f'{D}/{name}_rec.parquet')
    text = np.empty(int(rec.idx.max()) + 1, dtype=object); text[rec.idx.values] = rec.text.values
    pr = pd.read_parquet(f'{D}/{name}_pairs.parquet')
    return text, pr
def collate(batch, text):
    a = [text[x[0]] for x in batch]; b = [text[x[1]] for x in batch]
    enc = tok(a, b, truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')
    if len(batch[0]) > 2:
        enc['labels'] = torch.tensor([x[2] for x in batch], dtype=torch.float32)
    return enc

In [ ]:
# ---------- training (skipped if a finished model is already saved) ----------
if not os.path.exists(f'{CK}/DONE'):
    text, pr = load('train')
    pr = pr.sample(frac=1.0, random_state=0)
    items = list(zip(pr.a.values, pr.b.values, pr.label.values))
    print('train pairs', len(items), 'positives', pr.label.mean())
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).cuda()
    BS = 128
    dl = DataLoader(items, batch_size=BS, shuffle=False, num_workers=2, collate_fn=lambda b: collate(b, text))
    opt = torch.optim.AdamW(model.parameters(), lr=4e-5, weight_decay=0.01)
    steps = len(dl)
    sch = get_linear_schedule_with_warmup(opt, 500, steps)
    scaler = torch.cuda.amp.GradScaler()
    lossf = torch.nn.BCEWithLogitsLoss()
    model.train(); t0 = time.time(); run = 0
    for i, enc in enumerate(dl):
        enc = {k: v.cuda(non_blocking=True) for k, v in enc.items()}
        y = enc.pop('labels')
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(**enc).logits.squeeze(-1)
        loss = lossf(out.float(), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        run = 0.98 * run + 0.02 * loss.item()
        if i % 500 == 0:
            el = (time.time() - t0) / 60
            print(f'step {i}/{steps}  loss {run:.4f}  {el:.1f} min  {(i+1)*BS/max(el*60,1):.0f} pairs/s', flush=True)
        if (time.time() - t0) / 60 > TRAIN_MINUTES:
            print('time budget reached at step', i); break
    model.save_pretrained(CK); tok.save_pretrained(CK)
    open(f'{CK}/DONE', 'w').write('ok')
    del model; torch.cuda.empty_cache()
print('model ready')

In [ ]:
# ---------- scoring: holdout (for blending) and test, saved in chunks so it can resume ----------
model = AutoModelForSequenceClassification.from_pretrained(CK).cuda().half().eval()
def score(name, CH=500_000):
    final = f'{D}/{name}_ce.npy'
    if os.path.exists(final): print(name, 'done'); return
    text, pr = load(name)
    a, b = pr.a.values, pr.b.values
    n = len(pr); out = []
    for c in range(0, n, CH):
        f = f'{D}/{name}_ce_{c//CH:03d}.npy'
        if os.path.exists(f): out.append(np.load(f)); continue
        idx = np.arange(c, min(c + CH, n))
        ln = np.array([len(text[a[i]]) + len(text[b[i]]) for i in idx])
        order = idx[np.argsort(ln)]
        items = list(zip(a[order], b[order]))
        dl = DataLoader(items, batch_size=1024, num_workers=2, collate_fn=lambda x: collate(x, text))
        res = []
        with torch.no_grad():
            for enc in dl:
                enc = {k: v.cuda(non_blocking=True) for k, v in enc.items()}
                res.append(model(**enc).logits.squeeze(-1).float().cpu().numpy())
        s = np.empty(len(idx), np.float32); s[order - c] = np.concatenate(res)
        np.save(f, s.astype(np.float16)); out.append(s.astype(np.float16))
        print(name, f'{min(c+CH,n)}/{n}', time.strftime('%H:%M:%S'), flush=True)
    np.save(final, np.concatenate(out))
    print(name, 'saved', final)
score('hold')
score('test')

In [ ]:
# ---------- download the two result files ----------
from google.colab import files
files.download(f'{D}/hold_ce.npy'); files.download(f'{D}/test_ce.npy')